# Translating Poetry with Local LLMs (Colab version)

How do different language models translate the same poem? Does it matter whether the poem is world-famous or relatively obscure? Does "turning up the creativity" (temperature) change the result?

This notebook installs [Ollama](https://ollama.com) directly inside Colab so you can run open-source LLMs without any local setup.

## Setup — Install Ollama

**On GitHub Codespaces:** Ollama and models are pre-installed. This cell just starts the server (~5 seconds).

**On Google Colab:** This cell installs everything from scratch. It takes 2–3 minutes.

> We use **llama3** (8B) and **mistral** (7B). Colab's free tier has ~12 GB RAM, which is enough for one 7–8B model at a time. If you have Colab Pro (GPU + more RAM), you can uncomment the extra models below.

In [ ]:
%%bash
set -e

# Install Ollama if not already present (Codespaces has it pre-installed)
if ! command -v ollama &> /dev/null; then
    echo "Installing zstd and Ollama..."
    apt-get update -qq && apt-get install -y -qq zstd > /dev/null
    curl -fsSL https://ollama.com/install.sh | sh
fi

# Start the server if not already running
if ! curl -s http://127.0.0.1:11434/api/tags &> /dev/null; then
    echo "Starting Ollama server..."
    nohup ollama serve > /dev/null 2>&1 &
    sleep 3
fi

# Pull models only if not already downloaded
for model in llama3 mistral; do
    if ! ollama list | grep -q "^$model"; then
        echo "Pulling $model..."
        ollama pull "$model"
    else
        echo "$model already available"
    fi
done

# Uncomment for Colab Pro / more RAM:
# ollama pull phi4-mini
# ollama pull aya:8b

echo "Ready!"

## The poems

| Poem | Language | Translate to | Why? |
|------|----------|-------------|------|
| **Emily Dickinson** — *I heard a Fly buzz* (1862) | English | Spanish | Canonical — LLMs have seen many published translations |
| **Cesar Vallejo** — *Piedra negra sobre una piedra blanca* (1937) | Spanish | English | Major poet, far fewer English translations in training data |
| **Gabriela Mistral** — *Meciendo* (1922) | Spanish | English | Nobel laureate, but less translated; the verb *mecer* has no clean English equivalent |
| **Matsuo Basho** — *Furu ike ya* (1686) | Japanese | English | Most famous haiku ever — hundreds of competing translations |

In [ ]:
import json
import re
import requests
import textwrap
from itertools import product
from IPython.display import display, Markdown

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

POEMS = {
    "dickinson": {
        "title": "I heard a Fly buzz — when I died (Emily Dickinson, 1862)",
        "source_lang": "English",
        "target_lang": "Spanish",
        "text": textwrap.dedent("""\
            I heard a Fly buzz – when I died –
            The Stillness in the Room
            Was like the Stillness in the Air –
            Between the Heaves of Storm –"""),
        "note": "Canonical English poem — many published translations exist.",
    },
    "vallejo": {
        "title": "Piedra negra sobre una piedra blanca (César Vallejo, 1937)",
        "source_lang": "Spanish",
        "target_lang": "English",
        "text": textwrap.dedent("""\
            Me moriré en París con aguacero,
            un día del cual tengo ya el recuerdo.
            Me moriré en París — y no me corro —
            tal vez un jueves, como es hoy, de otoño."""),
        "note": "Major Peruvian poet, far fewer English translations in training data.",
    },
    "mistral_poet": {
        "title": "Meciendo (Gabriela Mistral, 1922)",
        "source_lang": "Spanish",
        "target_lang": "English",
        "text": textwrap.dedent("""\
            El mar sus millares de olas
            mece, divino.
            Oyendo a los mares amantes,
            mezo a mi niño."""),
        "note": "Chilean Nobel laureate, but less translated than Dickinson. "
                "The verb 'mecer' (to rock/cradle) has no clean English equivalent.",
    },
    "basho": {
        "title": "Furu ike ya (Matsuo Bashō, 1686)",
        "source_lang": "Japanese (romanised)",
        "target_lang": "English",
        "text": textwrap.dedent("""\
            Furu ike ya
            kawazu tobikomu
            mizu no oto"""),
        "note": "Most famous haiku ever — hundreds of competing translations exist.",
    },
}

# Print the originals
for key, poem in POEMS.items():
    print(f"--- {poem['title']} ---")
    print(f"({poem['source_lang']} → {poem['target_lang']})")
    print(poem["text"])
    print()

## The models

| Model | Parameters | Character |
|-------|-----------|-----------|
| **llama3** | 8B | Meta's general-purpose model — strong English, decent multilingual |
| **mistral** | 7B | French lab (Mistral AI) — strong European languages |

> On Colab Pro, uncomment `phi4-mini` and `aya:8b` in the setup cell above for more comparisons.

## What is temperature?

- **Low (0.2)** — the model picks the most likely word. Translations are **literal and safe**.
- **High (1.2)** — the model explores less obvious words. Translations are **more creative and varied**.

In [ ]:
# Adjust this list based on which models you pulled above
MODELS = ["llama3", "mistral"]
TEMPERATURES = [0.2, 1.2]

def make_prompt(poem):
    return (
        f"Translate the following poem from {poem['source_lang']} "
        f"into {poem['target_lang']}. "
        f"Preserve the poetic register and line breaks. "
        f"Output ONLY the translated poem, nothing else.\n\n"
        f"{poem['text']}"
    )

def generate(model, prompt, temperature=0.7):
    resp = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 512},
    }, timeout=300)
    resp.raise_for_status()
    text = resp.json()["response"].strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text

## Run the translations

This takes a few minutes — each model translates each poem at two temperatures.

In [ ]:
results = []
total = len(POEMS) * len(MODELS) * len(TEMPERATURES)
done = 0

for poem_key, poem in POEMS.items():
    prompt = make_prompt(poem)
    for model, temp in product(MODELS, TEMPERATURES):
        done += 1
        print(f"[{done}/{total}] {poem_key} | {model} | temp={temp} ... ", end="", flush=True)
        try:
            translation = generate(model, prompt, temperature=temp)
        except Exception as e:
            translation = f"[ERROR: {e}]"
        print("done")

        results.append({
            "poem": poem_key,
            "title": poem["title"],
            "source_lang": poem["source_lang"],
            "target_lang": poem["target_lang"],
            "original": poem["text"],
            "note": poem["note"],
            "model": model,
            "temperature": temp,
            "translation": translation,
        })

print(f"\nDone — {len(results)} translations generated.")

## Compare the translations

In [ ]:
for poem_key, poem in POEMS.items():
    md = f"### {poem['title']}\n"
    md += f"*{poem['source_lang']} → {poem['target_lang']}* — {poem['note']}\n\n"
    md += f"**Original:**\n```\n{poem['text']}\n```\n\n"

    poem_results = [r for r in results if r["poem"] == poem_key]

    md += "| Model | temp=0.2 | temp=1.2 |\n"
    md += "|-------|----------|----------|\n"
    for model in MODELS:
        row = f"| **{model}** "
        for temp in TEMPERATURES:
            match = [r for r in poem_results
                     if r["model"] == model and r["temperature"] == temp]
            if match:
                text = match[0]["translation"].replace("\n", "<br>")
                row += f"| {text} "
            else:
                row += "| — "
        md += row + "|\n"

    display(Markdown(md))
    print()

## Discussion — What to look for

**1. Culture and canonicity.** Compare the Dickinson translations (famous, many references) to Vallejo or Mistral (niche, fewer references). Are the Dickinson translations more uniform across models?

**2. Temperature.** For the same model and poem, compare temp=0.2 vs temp=1.2. At low temperature, is the rendering safe and literal? At high temperature, does the model take creative liberties — and are they *good*?

**3. Model training.** Does **mistral** (built by a French lab) show any advantage with European languages compared to **llama3**?

**4. Basho's haiku.** Do the models converge on a known version, or does each produce something original?

**5. What is lost.** Vallejo's *"no me corro"* is a pun. Mistral's *"mecer/mezo"* shifts from the cosmic (the sea rocking its waves) to the intimate (a mother rocking her child) — does any model capture that the same verb binds both?